<a href="https://colab.research.google.com/github/junaedifahmi/sft-lora-junaedifahmi/blob/main/sft_lora_bahasa_daerah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Fine-Tuning (SFT) dengan LoRA
## Studi Kasus: Bahasa Daerah Indonesia

**Tujuan:** Fine-tune small language model agar dapat mengikuti instruksi dalam bahasa daerah Indonesia.

**Stack:**
- `transformers` — load model & tokenizer
- `peft` — LoRA implementation
- `trl` — SFTTrainer
- `datasets` — load & process data

> ⚠️ Tidak menggunakan Unsloth atau layanan as-a-service. Semua dibangun dari library standar HuggingFace.

## 0. Instalasi Dependencies

In [ ]:
# Jalankan sekali saja di awal
# Di Colab: hapus tanda '#' di bawah
# !pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 datasets accelerate bitsandbytes

## 1. Import dan Konfigurasi

In [ ]:
import os
import json
import torch
from dataclasses import dataclass, field
from typing import List, Dict, Optional

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Konfigurasi Global

Ubah nilai di bawah sesuai pilihan Anda. Tidak perlu menyentuh kode lain.

In [ ]:
# ============================================================
#   KONFIGURASI — ubah sesuai kebutuhan
# ============================================================

# --- Pilihan model (uncomment satu) ---
MODEL_ID = "HuggingFaceTB/SmolLM2-360M"          # ~360M param, sangat cepat
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # ~1.1B param
# MODEL_ID = "Qwen/Qwen2.5-0.5B"                   # ~0.5B param, performa bagus
# MODEL_ID = "google/gemma-2-2b"                   # ~2B param, butuh akses HF
# MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"    # ~3.8B param, pakai 4-bit

# --- Pilihan dataset ---
DATASET_CHOICE = "synthetic"   # 'synthetic' | 'nusax' | 'cendol' | 'bryandts'

# --- Direktori output ---
OUTPUT_DIR   = "./output/sft-lora"
ADAPTER_DIR  = "./output/adapter"
MERGED_DIR   = "./output/merged-model"

# --- Hyperparameter pelatihan ---
MAX_SEQ_LEN     = 512
NUM_EPOCHS      = 2
BATCH_SIZE      = 2       # turunkan ke 1 jika OOM
GRAD_ACCUM      = 4       # efektif batch size = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE   = 2e-4
MAX_TRAIN_SAMPLES = 1000  # None = pakai semua

# --- LoRA hyperparameter ---
LORA_R      = 8
LORA_ALPHA  = 16
LORA_DROPOUT = 0.05

# --- Kuantisasi (aktifkan untuk model >=2B di GPU < 8GB) ---
USE_4BIT = False

# Seed untuk reproducibility
SEED = 42
torch.manual_seed(SEED)

## 3. Bagian Eksplorasi: Mengapa Base Model Lemah dalam Mengikuti Instruksi?

> **Pertanyaan kunci:** Model yang dilatih pada triliunan token teks internet memiliki pengetahuan luas, tapi mengapa gagal menjawab instruksi sederhana?

In [ ]:
# Muat tokenizer dan base model terlebih dahulu
print(f"Memuat model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Pastikan tokenizer punya pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocabulary size : {tokenizer.vocab_size:,}")
print(f"EOS token       : {tokenizer.eos_token!r}")
print(f"PAD token       : {tokenizer.pad_token!r}")

In [ ]:
# Konfigurasi kuantisasi (opsional)
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

total_params  = sum(p.numel() for p in base_model.parameters())
print(f"Total parameter : {total_params:,} ({total_params/1e6:.1f}M)")

In [ ]:
def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 200) -> str:
    """Helper: generate teks dari prompt."""
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    # Hanya ambil bagian yang digenerate (bukan prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# ============================================================
#   EKSPLORASI BASE MODEL — amati perilakunya sebelum fine-tuning
# ============================================================
test_prompts = [
    "Apa makna filosofi Jawa 'Memayu Hayuning Bawana'?",
    "Bagaimana cara membuat gudeg? Jelaskan langkah-langkahnya.",
    "Ceritakan legenda Sangkuriang dari Jawa Barat.",
]

print("=" * 60)
print("RESPONS BASE MODEL (sebelum fine-tuning)")
print("=" * 60)
for prompt in test_prompts:
    print(f"\n[INSTRUKSI]: {prompt}")
    response = generate_text(base_model, tokenizer, prompt)
    print(f"[RESPONS   ]: {response[:300]}")
    print("-" * 60)

print("""
ANALISIS:
Base model dilatih dengan objective 'next token prediction' pada teks mentah.
Ia belajar melanjutkan teks, BUKAN merespons instruksi.
Akibatnya: model cenderung melanjutkan kalimat tanya, bukan menjawabnya.
""")

## 4. Persiapan Dataset

### Pilihan Dataset Bahasa Daerah Indonesia

| Dataset | HuggingFace ID | Bahasa | Format |
|---------|---------------|--------|--------|
| **Cendol Collection v2** | `indonlp/cendol_collection_v2` | 10+ regional (Jawa, Sunda, Batak, Bugis...) | instruction-output |
| **Bryandts Instruct** | `bryandts/instruction-dataset-indo-java-sunda-bali-gayo-batak-alas-minang-betawi` | 9 bahasa daerah | instruction-input-output |
| **NusaX-MT** | `indonlp/NusaX-MT` | 12 bahasa (ID + 10 regional) | parallel translation |
| **NusaX-Senti** | `SEACrowd/nusax_senti` | 12 bahasa | sentiment classification |
| **Javanese Stories** | `Iftitahu/javanese_instruct_stories` | Jawa-Indonesia-Inggris | translation instruction |
| **IndoNLG** | `GEM/indonlg` | ID, Jawa, Sunda | QA, summarization, MT, dialog |
| **Korpus Nusantara** | `SEACrowd/korpus_nusantara` | 25 dialek | parallel translation |
| **Cendol v1** | `indonlp/cendol_collection_v1` | 10+ regional | task-specific instruction |

> **Rekomendasi:** Gunakan `bryandts/...` atau `indonlp/cendol_collection_v2` untuk SFT langsung karena sudah dalam format instruksi.

In [ ]:
# ============================================================
#   DATASET SINTETIS BAHASA DAERAH (built-in, tanpa download)
#   Gunakan ini jika tidak ada koneksi internet
# ============================================================

SYNTHETIC_DATA = [
    # --- Adat & Tradisi Jawa ---
    {
        "instruction": "Apa makna filosofi Jawa 'Memayu Hayuning Bawana'?",
        "input": "",
        "output": "Memayu Hayuning Bawana adalah filosofi Jawa yang berarti 'memperindah keindahan dunia'. Filosofi ini mengajarkan bahwa setiap manusia memiliki tanggung jawab untuk menjaga keselarasan, keharmonisan, dan keindahan alam semesta. Manusia tidak hanya hidup untuk dirinya sendiri, tetapi juga untuk merawat bumi dan sesama makhluk hidup."
    },
    {
        "instruction": "Jelaskan upacara adat Sunda 'Seren Taun' dan makna di baliknya.",
        "input": "",
        "output": "Seren Taun adalah upacara adat masyarakat Sunda sebagai bentuk syukur atas hasil panen padi. Kata 'seren' berarti menyerahkan dan 'taun' berarti tahun, sehingga maknanya adalah penyerahan hasil bumi selama setahun. Upacara ini biasanya dilaksanakan di Kasepuhan Ciptagelar, Sukabumi, dan melibatkan prosesi membawa padi dari sawah ke lumbung (leuit), diiringi musik angklung dan gamelan. Makna spiritualnya adalah penghormatan kepada Dewi Sri (dewi padi) dan ungkapan terima kasih kepada Tuhan."
    },
    {
        "instruction": "Apa itu 'Dalihan Na Tolu' dalam budaya Batak?",
        "input": "",
        "output": "Dalihan Na Tolu adalah sistem kekerabatan tiga tungku dalam budaya Batak. Tiga unsurnya adalah: (1) Hula-hula — pihak keluarga pemberi istri (mertua), yang dihormati tertinggi; (2) Dongan Tubu — saudara semarga, yang menjadi teman seperjuangan; (3) Boru — pihak keluarga penerima istri (menantu), yang bertanggung jawab melayani. Sistem ini mengatur seluruh hubungan sosial, adat, dan gotong royong dalam masyarakat Batak. Filosofinya: tidak ada yang boleh terlalu tinggi atau terlalu rendah, semua saling membutuhkan."
    },
    {
        "instruction": "Ceritakan secara singkat legenda Sangkuriang dari Jawa Barat.",
        "input": "",
        "output": "Sangkuriang adalah legenda asal-usul Gunung Tangkuban Parahu dari Jawa Barat. Dikisahkan, Dayang Sumbi memiliki anak bernama Sangkuriang yang diusir karena berbuat salah. Setelah bertahun-tahun mengembara, Sangkuriang kembali dan jatuh cinta pada Dayang Sumbi—yang tidak menua karena memakan hati anjing sakti. Dayang Sumbi menyadari bahwa pria itu adalah anaknya sendiri dan mencari cara untuk menolak lamaran. Ia memberi syarat mustahil: Sangkuriang harus membuat danau dan perahu dalam satu malam. Hampir berhasil, Dayang Sumbi menggagalkan usahanya dengan membuat fajar tiba lebih awal. Sangkuriang marah dan menendang perahunya hingga terbalik, menjadi Gunung Tangkuban Parahu."
    },
    {
        "instruction": "Apa perbedaan antara bahasa Jawa Ngoko dan Krama?",
        "input": "",
        "output": "Bahasa Jawa memiliki tingkatan (unggah-ungguh) yang mencerminkan sopan santun. Ngoko adalah tingkatan paling informal, digunakan antara teman sebaya atau orang yang lebih tua kepada yang lebih muda. Contoh: 'Kowe arep menyang ngendi?' (Kamu mau ke mana?). Krama adalah tingkatan formal, digunakan untuk menghormati lawan bicara yang lebih tua atau lebih tinggi kedudukannya. Contoh: 'Panjenengan badhe tindak dhateng pundi?' (Anda hendak pergi ke mana?). Ada juga Krama Inggil sebagai tingkatan paling hormat, dan Madya sebagai tingkatan menengah."
    },
    # --- Kuliner Nusantara ---
    {
        "instruction": "Bagaimana cara membuat rendang daging sapi? Jelaskan bahan dan langkahnya.",
        "input": "",
        "output": "Bahan utama: 1 kg daging sapi (potong dadu), 1 liter santan kental, 5 lembar daun jeruk, 2 batang serai, 3 lembar daun salam, 1 buah asam kandis.\n\nBumbu halus: 15 cabai merah kering, 10 bawang merah, 6 siung bawang putih, 3 cm jahe, 3 cm lengkuas, 1 sdt ketumbar, garam secukupnya.\n\nLangkah-langkah:\n1. Haluskan semua bumbu halus.\n2. Masukkan daging, santan, dan semua bumbu ke dalam wajan besar.\n3. Masak dengan api sedang sambil terus diaduk hingga mendidih.\n4. Kecilkan api, masak terus hingga santan mengering dan bumbu meresap (~3-4 jam).\n5. Terus aduk agar tidak gosong sampai rendang berwarna cokelat kehitaman.\n6. Sajikan dengan nasi putih hangat."
    },
    {
        "instruction": "Apa bahan-bahan khas yang digunakan dalam masakan Betawi?",
        "input": "",
        "output": "Masakan Betawi memiliki ciri khas penggunaan santan yang kental, rempah yang kuat, dan perpaduan budaya Melayu, Cina, Belanda, dan Arab. Bahan-bahan khas yang sering digunakan antara lain: (1) Oncom — hasil fermentasi bungkil kedelai, khas Betawi dan Sunda; (2) Petai dan jengkol — sering muncul dalam masakan berkuah; (3) Cabai rawit berlimpah untuk rasa pedas; (4) Terasi Bekasi berkualitas tinggi; (5) Kemiri untuk mengentalkan bumbu; (6) Daun kemangi segar sebagai lalapan. Hidangan ikonik Betawi antara lain Soto Betawi, Ketoprak, Gado-Gado, dan Kerak Telor."
    },
    {
        "instruction": "Jelaskan proses pembuatan tape singkong (peuyeum) khas Sunda.",
        "input": "",
        "output": "Peuyeum atau tape singkong adalah makanan fermentasi khas Sunda, terutama dari Bandung.\n\nBahan: 1 kg singkong (kupas, potong), 2 butir ragi tape.\n\nCara membuat:\n1. Kukus singkong hingga matang dan empuk, sekitar 30-40 menit.\n2. Angkat dan biarkan dingin di atas tampah atau nampan bersih.\n3. Haluskan ragi tape menjadi bubuk, taburkan merata di atas singkong yang sudah dingin.\n4. Susun singkong dalam wadah tertutup yang dilapisi daun pisang.\n5. Tutup rapat dan fermentasikan selama 2-3 hari di suhu ruang.\n6. Peuyeum siap disantap ketika teksturnya lembut, terasa manis-asam, dan sedikit berair.\n\nKunci sukses: singkong harus benar-benar dingin sebelum diberi ragi, dan wadah harus kedap udara."
    },
    # --- Instruksi Kehidupan Sehari-hari ---
    {
        "instruction": "Berikan 5 tips menjaga kesehatan menurut kearifan lokal Indonesia.",
        "input": "",
        "output": "1. **Jamu sebagai preventif** — Minum jamu tradisional seperti kunyit asam, beras kencur, atau temulawak secara rutin untuk menjaga daya tahan tubuh.\n2. **Makan sesuai musim** — Konsumsi buah dan sayur yang sedang musim karena segar dan bergizi tinggi.\n3. **Istirahat mengikuti siklus alam** — Tidur sebelum pukul 22.00 dan bangun sebelum subuh sesuai ajaran leluhur untuk menjaga ritme sirkadian.\n4. **Gerakan sederhana setiap hari** — Gotong royong, berkebun, atau sekadar berjalan kaki ke sawah sudah merupakan aktivitas fisik yang cukup.\n5. **Pikiran tenang dengan syukur** — Budaya bersyukur (ojo dumeh, nrimo ing pandum dalam filosofi Jawa) terbukti mengurangi stres dan menjaga kesehatan mental."
    },
    {
        "instruction": "Terjemahkan kalimat ini ke bahasa Jawa Krama: 'Saya ingin pergi ke pasar membeli sayuran'.",
        "input": "",
        "output": "Terjemahan dalam bahasa Jawa Krama: 'Kula badhe tindak dhateng peken mundhut sayuran.'\n\nPenjelasan:\n- 'Kula' = saya (Krama dari 'aku')\n- 'badhe' = ingin/akan (Krama dari 'arep')\n- 'tindak' = pergi (Krama Inggil, lebih hormat dari 'lunga')\n- 'dhateng' = ke (Krama dari 'menyang')\n- 'peken' = pasar (Krama dari 'pasar')\n- 'mundhut' = membeli (Krama dari 'tuku')"
    },
    {
        "instruction": "Apa saja pantangan (pamali) yang umum dalam budaya Sunda dan apa maknanya?",
        "input": "",
        "output": "Beberapa pamali (pantangan) dalam budaya Sunda dan makna filosofisnya:\n\n1. **Ulah dahar di lawang panto** (Jangan makan di depan pintu) — Maknanya: menghambat rezeki yang masuk ke rumah; secara praktis mencegah orang lain kesulitan melewati pintu.\n2. **Ulah hujan-hujanan** (Jangan kehujanan sembarangan) — Nasihat kesehatan: air hujan dari atap bisa mengandung kotoran.\n3. **Ulah nyapu peuting** (Jangan menyapu malam hari) — Secara praktis: di zaman dulu tanpa listrik, menyapu malam bisa membuang barang berharga. Secara simbolis: jangan membuang keberuntungan.\n4. **Ulah ngaliwatan nu keur dahar** (Jangan melewati orang yang sedang makan) — Bentuk sopan santun dan penghormatan.\n5. **Ulah nembang wayah peuting** (Jangan bernyanyi di malam hari) — Konon mengundang makhluk gaib; praktisnya mengganggu tetangga yang istirahat."
    },
    # --- Cerita Rakyat & Sejarah ---
    {
        "instruction": "Siapa tokoh Si Pitung dan mengapa ia dianggap pahlawan rakyat Betawi?",
        "input": "",
        "output": "Si Pitung adalah tokoh legendaris Betawi yang hidup di akhir abad ke-19, pada masa penjajahan Belanda di Batavia (Jakarta). Ia dikenal sebagai 'robinhood Betawi' karena dikisahkan merampok dari orang kaya dan penjajah Belanda, lalu membagikan hasilnya kepada rakyat miskin.\n\nAlasan ia dianggap pahlawan rakyat:\n1. Berani melawan ketidakadilan kolonial saat rakyat jelata tertindas.\n2. Memiliki ilmu bela diri dan kebal senjata (menurut legenda).\n3. Berpihak pada kaum lemah dan miskin.\n4. Menjadi simbol perlawanan masyarakat Betawi terhadap penguasa asing.\n\nMeski terdapat perdebatan sejarah tentang kenyataan kisahnya, Si Pitung tetap menjadi ikon budaya Betawi yang dilestarikan dalam film, teater lenong, dan cerita turun-temurun."
    },
    {
        "instruction": "Jelaskan sistem 'Subak' di Bali dan mengapa diakui UNESCO.",
        "input": "",
        "output": "Subak adalah sistem irigasi tradisional Bali yang telah ada sejak abad ke-9 Masehi. Ia merupakan organisasi kemasyarakatan yang mengatur pembagian air untuk sawah-sawah di Bali berdasarkan prinsip 'Tri Hita Karana' (tiga penyebab kebaikan): keselarasan antara manusia dengan Tuhan, manusia dengan sesama, dan manusia dengan alam.\n\nCara kerja Subak:\n- Petani tergabung dalam kelompok subak yang diketuai Pekaseh.\n- Air dari sumber mata air atau sungai dibagi adil melalui saluran-saluran tradisional (telabah).\n- Keputusan diambil secara musyawarah di pura subak.\n- Ritual keagamaan dilakukan sebelum dan sesudah panen sebagai ungkapan syukur.\n\nUNESCO mengakui Subak sebagai Warisan Budaya Dunia pada 2012 karena merupakan contoh luar biasa sistem pertanian berkelanjutan yang memadukan spiritualitas, ekologi, dan demokrasi lokal."
    },
    # --- Tambahan contoh lebih bervariasi ---
    {
        "instruction": "Buatkan pantun Melayu tentang semangat belajar.",
        "input": "",
        "output": "Berikut pantun Melayu tentang semangat belajar:\n\nBuah manggis di tepi kali,\nDiambil orang waktu pagi.\nRajin belajar dari kecil,\nHidup mulia sampai mati.\n\n---\n\nPohon kelapa tumbuh di pantai,\nBuahnya lebat sangat banyak.\nIlmu yang tinggi jadikan lantai,\nUntuk membangun bangsa yang bijak."
    },
    {
        "instruction": "Apa itu 'Marsipature Hutana Be' dalam budaya Batak Toba?",
        "input": "",
        "output": "'Marsipature Hutana Be' (disingkat MHB) adalah slogan dan gerakan yang berarti 'Membangun Kampung Halaman' dalam bahasa Batak Toba. Konsep ini mengajak masyarakat Batak yang merantau ke kota-kota besar untuk tetap berkontribusi membangun kampung halaman mereka di Sumatera Utara.\n\nNilai-nilai yang terkandung:\n1. **Hamoraon** (kekayaan) — didapat bukan hanya untuk diri sendiri, tapi dibawa pulang membangun kampung.\n2. **Hasangapon** (kehormatan) — nama baik dijaga dengan berkontribusi nyata bagi tanah Batak.\n3. **Hagabeon** (keturunan) — memastikan generasi berikutnya tetap terhubung dengan akar budayanya.\n\nGerakan ini sering disuarakan oleh tokoh-tokoh Batak yang sukses di perantauan sebagai bentuk tanggung jawab terhadap komunitas asal."
    },
    {
        "instruction": "Bagaimana cara memainkan alat musik angklung?",
        "input": "",
        "output": "Angklung adalah alat musik tradisional Sunda yang terbuat dari bambu dan diakui UNESCO sebagai Warisan Budaya Takbenda (2010).\n\nCara memainkan:\n1. **Memegang angklung** — Pegang bingkai bambu dengan tangan kiri di bagian atas, sehingga angklung dapat bergerak bebas.\n2. **Menggoyangkan** — Goyang angklung dengan gerakan tangan kanan ke kiri dan ke kanan secara horizontal. Setiap angklung hanya menghasilkan satu nada.\n3. **Bermain bersama** — Karena satu angklung = satu nada, musik angklung selalu dimainkan berkelompok. Setiap pemain bertanggung jawab atas nada(s) tertentu.\n4. **Teknik dasar:** Kurulung (goyangan berulang cepat untuk nada panjang), Centok (pukulan singkat untuk nada pendek).\n\nAngklung tersedia dalam skala diatonis, pentatonis, dan kromatis, sehingga dapat memainkan berbagai jenis lagu."
]

print(f"Dataset sintetis: {len(SYNTHETIC_DATA)} contoh")
print(f"Contoh pertama:")
print(f"  Instruksi : {SYNTHETIC_DATA[0]['instruction']}")
print(f"  Output    : {SYNTHETIC_DATA[0]['output'][:80]}...")

In [ ]:
def load_dataset_hf(choice: str, max_samples: Optional[int] = None) -> List[Dict]:
    """Muat dataset dari HuggingFace dan konversi ke format instruksi standar."""
    from datasets import load_dataset

    records = []

    if choice == "nusax":
        # NusaX-MT: terjemahan Indonesia <-> bahasa daerah
        # Format: Jadikan instruksi terjemahan
        ds = load_dataset("indonlp/NusaX-MT", "ind-jav", split="train")  # Indonesia-Jawa
        for row in ds:
            records.append({
                "instruction": f"Terjemahkan kalimat berikut dari bahasa Indonesia ke bahasa Jawa: {row['ind']}",
                "input": "",
                "output": row["jav"],
            })

    elif choice == "cendol":
        # Cendol Collection v2: sudah dalam format instruksi
        ds = load_dataset("indonlp/cendol_collection_v2", split="train")
        for row in ds:
            records.append({
                "instruction": row.get("input", ""),
                "input": "",
                "output": row.get("output", ""),
            })

    elif choice == "bryandts":
        # Bryandts: format Alpaca (instruction, input, output)
        ds = load_dataset(
            "bryandts/instruction-dataset-indo-java-sunda-bali-gayo-batak-alas-minang-betawi",
            split="train"
        )
        for row in ds:
            records.append({
                "instruction": row["instruction"],
                "input": row.get("input", ""),
                "output": row["output"],
            })

    else:
        raise ValueError(f"Dataset tidak dikenal: {choice}")

    if max_samples:
        import random
        random.seed(SEED)
        records = random.sample(records, min(max_samples, len(records)))

    return records


# Pilih sumber data
if DATASET_CHOICE == "synthetic":
    raw_data = SYNTHETIC_DATA
    print(f"Menggunakan dataset sintetis: {len(raw_data)} contoh")
else:
    print(f"Memuat dataset dari HuggingFace: {DATASET_CHOICE}")
    raw_data = load_dataset_hf(DATASET_CHOICE, max_samples=MAX_TRAIN_SAMPLES)
    print(f"Jumlah data dimuat: {len(raw_data)}")

In [ ]:
# ============================================================
#   FORMAT KE CHAT TEMPLATE
#   Setiap contoh dikonversi ke format messages (role/content)
#   agar sesuai dengan chat template tokenizer
# ============================================================

def format_to_chat(record: Dict) -> Dict:
    """Konversi record {instruction, input, output} ke format chat messages."""
    user_content = record["instruction"]
    if record.get("input", "").strip():
        user_content = f"{user_content}\n\nKonteks:\n{record['input']}"

    return {
        "messages": [
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": record["output"]},
        ]
    }


formatted = [format_to_chat(r) for r in raw_data]

# Terapkan chat template tokenizer
def apply_chat_template(examples):
    texts = []
    for msgs in examples["messages"]:
        try:
            text = tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=False,
            )
        except Exception:
            # Fallback manual jika tokenizer tidak punya chat template
            user = msgs[0]["content"]
            asst = msgs[1]["content"]
            text = f"### Instruksi:\n{user}\n\n### Respons:\n{asst}{tokenizer.eos_token}"
        texts.append(text)
    return {"text": texts}


# Buat HuggingFace Dataset
hf_dataset = Dataset.from_list(formatted)
hf_dataset = hf_dataset.map(apply_chat_template, batched=True, remove_columns=["messages"])

# Train/val split 90/10
split = hf_dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train : {len(train_dataset)} contoh")
print(f"Eval  : {len(eval_dataset)} contoh")
print("\nContoh teks setelah format:")
print(train_dataset[0]["text"][:400])

## 5. Konfigurasi LoRA

LoRA menambahkan matriks berdimensi rendah $\Delta W = B \cdot A$ pada layer attention.
Hanya parameter $A$ dan $B$ yang dilatih, bukan bobot asli model.

In [ ]:
# Target modules bervariasi per arsitektur model
TARGET_MODULES_MAP = {
    "SmolLM2"  : ["q_proj", "k_proj", "v_proj", "o_proj"],
    "TinyLlama": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "Qwen2"    : ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "gemma"    : ["q_proj", "k_proj", "v_proj", "o_proj"],
    "Phi-3"    : ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
}

# Deteksi arsitektur otomatis
arch = base_model.config.architectures[0] if base_model.config.architectures else ""
target_modules = ["q_proj", "v_proj"]  # default aman
for key, modules in TARGET_MODULES_MAP.items():
    if key.lower() in arch.lower() or key.lower() in MODEL_ID.lower():
        target_modules = modules
        break

print(f"Arsitektur   : {arch}")
print(f"Target modules: {target_modules}")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=target_modules,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Terapkan LoRA pada model
model = get_peft_model(base_model, lora_config)

# Hitung parameter yang dapat dilatih
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nParameter dilatih : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Parameter total   : {total:,}")
print(f"Parameter frozen  : {total - trainable:,} ({100*(total-trainable)/total:.2f}%)")

## 6. Pelatihan dengan SFTTrainer

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

sft_config = SFTConfig(
    # Output
    output_dir=OUTPUT_DIR,

    # Epoch & batch
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Memori
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Presisi
    fp16=torch.cuda.is_available() and not USE_4BIT,
    bf16=False,

    # Optimizer
    optim="adamw_torch",
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    # Sekuens
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,  # True untuk dataset kecil (efisiensi), False untuk dataset besar

    # Logging & Eval
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Reproducibility
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("Konfigurasi pelatihan siap.")
print(f"  Efektif batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total steps        : {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS}")

In [ ]:
# ============================================================
#   MULAI PELATIHAN
# ============================================================
print("Memulai pelatihan...")
train_result = trainer.train()

print("\n--- Hasil Pelatihan ---")
print(f"Train loss akhir  : {train_result.training_loss:.4f}")
print(f"Total steps       : {train_result.global_step}")
print(f"Waktu pelatihan   : {train_result.metrics.get('train_runtime', 0):.1f} detik")

In [ ]:
# Simpan adapter LoRA (ringan — hanya bobot delta, bukan full model)
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter LoRA disimpan di: {ADAPTER_DIR}")

# Tampilkan ukuran file
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Ukuran adapter    : {total_size / 1e6:.1f} MB")

## 7. Evaluasi Kualitatif

Bandingkan respons model **sebelum** dan **sesudah** fine-tuning pada prompt yang sama.

In [ ]:
# Muat ulang base model untuk perbandingan
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Muat model yang sudah di-fine-tune
finetuned_model = PeftModel.from_pretrained(base_model_eval, ADAPTER_DIR)
finetuned_model.eval()
print("Model fine-tuned berhasil dimuat.")

In [ ]:
# Prompt uji (di luar dataset pelatihan)
eval_prompts = [
    "Jelaskan makna filosofi Bugis 'Siri Na Pacce' dan relevansinya dalam kehidupan modern.",
    "Apa perbedaan antara wayang kulit dan wayang golek?",
    "Bagaimana cara membuat opor ayam? Sebutkan bahan dan langkahnya.",
    "Terjemahkan ke bahasa Sunda: 'Saya sangat senang bertemu dengan Anda hari ini.'",
    "Ceritakan legenda asal-usul Danau Toba.",
]

print("=" * 70)
print("PERBANDINGAN: BASE MODEL vs MODEL FINE-TUNED")
print("=" * 70)

for i, prompt in enumerate(eval_prompts, 1):
    print(f"\n{'='*70}")
    print(f"[Prompt {i}]: {prompt}")
    print("-" * 70)

    resp_base = generate_text(base_model_eval, tokenizer, prompt, max_new_tokens=150)
    print(f"[BASE MODEL]:\n{resp_base[:300]}")
    print()

    resp_ft = generate_text(finetuned_model, tokenizer, prompt, max_new_tokens=150)
    print(f"[FINE-TUNED]:\n{resp_ft[:300]}")

## 8. Visualisasi Training Loss

In [ ]:
import matplotlib.pyplot as plt

# Ambil log history dari trainer
log_history = trainer.state.log_history

train_steps  = [x["step"] for x in log_history if "loss" in x]
train_losses = [x["loss"] for x in log_history if "loss" in x]
eval_steps   = [x["step"] for x in log_history if "eval_loss" in x]
eval_losses  = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_losses, label="Train Loss", color="steelblue", linewidth=2)
if eval_losses:
    ax.plot(eval_steps, eval_losses, label="Eval Loss", color="tomato",
            linestyle="--", marker="o", linewidth=2)
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title(f"Training Curve — {MODEL_ID.split('/')[-1]}", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafik disimpan sebagai 'training_curve.png'")

## 9. (Opsional) Merge LoRA ke Base Model

Jika ingin menyimpan model lengkap (tanpa adapter terpisah) untuk deployment:

In [ ]:
# Merge adapter ke dalam bobot base model
# Catatan: membutuhkan RAM/VRAM lebih besar

MERGE_MODEL = False  # Ubah ke True jika ingin merge

if MERGE_MODEL:
    print("Merge adapter ke base model...")
    merged = finetuned_model.merge_and_unload()
    os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    print(f"Model merged disimpan di: {MERGED_DIR}")
else:
    print("Skip merge. Gunakan adapter terpisah saat inference.")
    print("Untuk load saat inference:")
    print("  base = AutoModelForCausalLM.from_pretrained(MODEL_ID)")
    print(f"  model = PeftModel.from_pretrained(base, '{ADAPTER_DIR}')")

## 10. Pertanyaan Analisis

Jawab pertanyaan berikut dalam laporan Anda:

1. **Berapa persen parameter yang dilatih dengan LoRA?** (lihat output sel 5)

2. **Mengapa base model gagal mengikuti instruksi?** Jelaskan dari sudut pandang fungsi loss saat pre-training.

3. **Bandingkan 3 contoh respons sebelum dan sesudah fine-tuning.** Apa perubahan yang Anda amati?

4. **Apakah ada tanda-tanda overfitting?** Bagaimana Anda mendeteksinya dari grafik loss?

5. **(Bonus)** Jika Anda membuat dataset bahasa daerah sendiri: apa tantangan terbesar dalam pembuatan data berkualitas?

---
*Selamat mengerjakan dan bereksperimen!*